# Install

`onceover` is a Python package with a Rust core. Binary wheels are the normal
path; building from source needs a Rust toolchain.

```bash
pip install onceover
```

The package requires CPython 3.10+ and Polars 2.x. Wheels cover Linux x86_64
and aarch64, macOS Intel and Apple silicon, and Windows x86_64. The source
distribution additionally needs Rust and a C compiler — the same setup used
for development, below.

## Verify the installation

Verify the installation by importing `onceover` and profiling a small
DataFrame. This loads the compiled extension and runs the pipeline.

`onceover` can profile in-memory DataFrames with millions of rows and
hundreds of columns. Column summaries and integrity checks examine every
row. Relationship discovery uses bounded samples and searches; see
[](coverage.ipynb) for the limits and coverage fields.

In [ ]:
import polars as pl
import onceover

print(onceover.__version__)

df = pl.DataFrame({"id": [1, 2, 3], "value": [0.5, 1.5, None]})
report = onceover.profile(df, budget="quick")
print(report.overview["headline"])

## Optional dependencies

Two extras install what the surrounding tooling needs:

| Extra | Installs | Used by |
|---|---|---|
| `onceover[notebook]` | altair, great-tables, vl-convert, numpy, jupyterlab | the analysis notebook, charts, tables |
| `onceover[benchmark]` | numpy, psutil | the scale benchmark |

```bash
pip install 'onceover[notebook]'
```

## Development installation

Development builds the Rust core and binds it to a virtual environment:

```bash
python -m venv .venv
source .venv/bin/activate
pip install 'maturin>=1.15,<2' pytest scipy
maturin develop --release
cargo test --workspace --locked
pytest tests/
```

`maturin develop --release` compiles the workspace and installs the extension
into the active environment. Repeat it after changing Rust sources; the
Python side is plain source and needs no build step.

The profiler releases the GIL during computation. Multiple threads can
therefore call `profile` concurrently without serializing on the
interpreter.